# April 16 LaCT Fast-Weight Record Search on Colab

This notebook runs `colab/2026-04-16_LaCT_FastWeight_RecordSearch` on Google Colab with a quality-oriented preset.

The goal is to minimize final `val_bpb`, especially the logged `quantized_lact_ttt` metric. This is not a cheap smoke-test preset. It is configured to spend more Colab time for better quality while staying safer for a single Colab GPU than the full 8xH100 record profile.

In [ ]:
import subprocess, textwrap, os, pathlib

print(subprocess.check_output(['nvidia-smi'], text=True))

## Why these env vars are chosen

The April 16 experiment is based on the April 9 SP8192 record stack and adds LaCT-style fast-weight test-time training. The lowest-BPB path should keep the April 9 record ingredients intact, then add the strongest legal eval-time adaptation that can fit on Colab.

- `RECORD_PROFILE=0`: the full record profile uses a huge single-step batch and is intended for 8xH100. On a Colab T4 it is likely to OOM or make poor progress. The notebook keeps the single-GPU profile but overrides the quality-critical eval/export knobs.
- `MAX_WALLCLOCK_SECONDS=0`: the default 600s cap is useful for leaderboard timing, but on T4 it stops too early and severely hurts quality. For minimum `val_bpb` on Colab, disable the cap and let the selected `ITERATIONS` run.
- `ITERATIONS=5000`: this is a practical Colab compromise. It is much stronger than a 300-step smoke test, but less likely to exceed a Colab session than the full `20000` steps. If runtime allows, increase this first.
- `TRAIN_SHARDS=10`: keeps the Colab comparison surface aligned with the project convention and avoids downloading the much larger 128-shard record dataset view.
- `VOCAB_SIZE=8192`: keeps the April 9 SP8192 tokenizer stack, which is the current stronger leaderboard family.
- `QK_GAIN_INIT=5.25`: matches the April 9 winning direction, where QK gain above 5.0 was part of the best stack.
- `EXPORT_ALLOCATOR=entropy`: uses the April 14 entropy-constrained allocator instead of the fixed legacy export. This is most likely to improve byte allocation under the hard 16MB cap.
- `ALLOCATOR_MATRIX_BITS=5,6,7` and `ALLOCATOR_EMBED_BITS=7,8`: lets the allocator choose mixed bitwidths instead of forcing one global precision.
- `ALLOCATOR_MATRIX_SIGMAS=10.5,12.85,15.0` and `ALLOCATOR_EMBED_SIGMAS=16.0,20.0,24.0`: keeps the search centered around the April SDClip values while giving the allocator room to trade quality against compressed bytes.
- `LACT_TTT_ENABLED=1`: enables the new April 16 fast-weight eval path. This is the main experimental improvement over the April 9 baseline.
- `TTT_ENABLED=0` and `LACT_BASE_TTT=1`: follows the README's hybrid candidate. Legacy base-model TTT is not run as a separate metric path, but it is combined inside the LaCT eval path.
- `LACT_FAST_WEIGHT=swiglu`: the nonlinear adapter has more capacity than the linear fallback and is the primary record-search path.
- `LACT_UPDATE=muon`: uses the stronger Muon-style update for the fast weights rather than plain SGD.
- `LACT_STATE_DIM=128`: stronger than the T4 default 64 and matches the intended record setting. If you OOM during `quantized_lact_ttt`, lower this to 64.
- `LACT_BATCH_SEQS=4`: conservative for T4 memory. Increasing it usually improves throughput, not quality.
- `LACT_LR=0.02`, `LACT_SCALE=0.08`, `LACT_EPOCHS=1`, `LACT_CHUNK_TOKENS=32768`: these are the intended April 16 record-search defaults, kept unchanged because they define the current best prior for this experiment.

## Clone or update the repo

If you already cloned your fork into `/content/parameter-golf`, this cell updates it. If not, it clones it.

In [ ]:
%%bash
set -euo pipefail
cd /content
if [[ ! -d parameter-golf/.git ]]; then
  git clone https://github.com/IanniMuliterno/parameter-golf.git parameter-golf
else
  cd parameter-golf
  git fetch origin
  git pull --ff-only || true
fi
cd /content/parameter-golf/colab/2026-04-16_LaCT_FastWeight_RecordSearch
pwd
git rev-parse --short HEAD

## Quality-oriented Colab run

This is the main run cell. The expected key log lines are:

- `latest_valid_record ... ttt_bpb:1.0810 ...`
- `pre-quantization post-ema`
- `quantized`
- `quantized_sliding_window`
- `quantized_lact_ttt`
- `artifact bytes`

Use `quantized_lact_ttt` as the main number for this experiment. If Colab disconnects, reduce `ITERATIONS` first for debugging, then restore it for serious runs.

In [ ]:
%%bash
set -euo pipefail
cd /content/parameter-golf/colab/2026-04-16_LaCT_FastWeight_RecordSearch

INSTALL_DEPS=1 \
DOWNLOAD_DATA=1 \
TRAIN_SHARDS=10 \
NPROC_PER_NODE=1 \
RECORD_PROFILE=0 \
RUN_ID=apr16_colab_quality_lact_swiglu_muon_s128_base_ttt \
SEED=42 \
VOCAB_SIZE=8192 \
QK_GAIN_INIT=5.25 \
TRAIN_BATCH_TOKENS=65536 \
VAL_BATCH_TOKENS=65536 \
TRAIN_SEQ_LEN=2048 \
EVAL_SEQ_LEN=2048 \
ITERATIONS=5000 \
MAX_WALLCLOCK_SECONDS=600 \
VAL_LOSS_EVERY=0 \
TRAIN_LOG_EVERY=100 \
COMPUTE_DTYPE=fp16 \
ENABLE_COMPILE=0 \
PG_COLAB_DISABLE_COMPILE=1 \
COMPRESSOR=brotli \
EXPORT_ALLOCATOR=entropy \
ARTIFACT_TARGET_BYTES=16000000 \
ALLOCATOR_GROUP_COLS=128 \
ALLOCATOR_MATRIX_BITS=5,6,7 \
ALLOCATOR_EMBED_BITS=7,8 \
ALLOCATOR_MATRIX_SIGMAS=10.5,12.85,15.0 \
ALLOCATOR_EMBED_SIGMAS=16.0,20.0,24.0 \
ALLOCATOR_USE_ENTROPY_PROXY=1 \
ALLOCATOR_CODE_WRAPPERS=source,lzma_raw_b85_exec \
GPTQ_CALIBRATION_BATCHES=64 \
GPTQ_RESERVE_SECONDS=180 \
SLIDING_WINDOW_ENABLED=1 \
LACT_TTT_ENABLED=1 \
TTT_ENABLED=0 \
LACT_BASE_TTT=1 \
LACT_FAST_WEIGHT=swiglu \
LACT_UPDATE=muon \
LACT_STATE_DIM=128 \
LACT_BATCH_SEQS=4 \
LACT_CHUNK_TOKENS=32768 \
LACT_EPOCHS=1 \
LACT_LR=0.02 \
LACT_SCALE=0.08 \
LACT_MOMENTUM=0.9 \
LACT_GRAD_CLIP=1.0 \
LACT_NORMALIZE=1 \
bash run.sh

## If you get OOM

Change only one thing at a time. The least damaging reductions are:

1. `LACT_STATE_DIM=64`: lowers LaCT capacity but usually fixes eval memory.
2. `GPTQ_CALIBRATION_BATCHES=16`: lowers calibration quality but shortens export and reduces memory pressure.
3. `ITERATIONS=2000`: faster debugging, but not a serious minimum-BPB run.
4. `LACT_FAST_WEIGHT=linear`: last resort. It is cheaper but likely worse than `swiglu`.

## If you have Colab A100 or another larger GPU

For a stronger single-GPU run, keep the same quality preset but try:

```bash
COMPUTE_DTYPE=bf16
GPTQ_CALIBRATION_BATCHES=64
LACT_STATE_DIM=192
LACT_BATCH_SEQS=8
ITERATIONS=10000
```

I would still avoid `RECORD_PROFILE=1` on ordinary Colab unless you know the GPU has enough memory for the large `TRAIN_BATCH_TOKENS=786432` profile. That profile is intended for the 8xH100 record shape, not a default Colab T4.

## Show final metrics from the latest log

Run this after the training cell finishes.

In [ ]:
%%bash
set -euo pipefail
cd /content/parameter-golf/colab/2026-04-16_LaCT_FastWeight_RecordSearch
LOG="logs/apr16_colab_quality_lact_swiglu_muon_s128_base_ttt.txt"
if [[ ! -f "$LOG" ]]; then
  LOG="$(ls -t logs/*.txt | head -1)"
fi
echo "Using log: $LOG"
grep -E "latest_valid_record|pre-quantization post-ema|quantized|quantized_sliding_window|quantized_lact_ttt|artifact bytes|allocator_selected" "$LOG" | tail -80